# Data cleaning pipeline

This notebook builds MatchCast's processed match table from the raw
`martj42/international_results` snapshot. It is built up section by section,
one cleaning concern at a time, matching the schema defined in
`01_processed_match_schema.ipynb`. Each section documents its rule and
validates it before the next section runs.

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/international_results.csv")
PROCESSED_PATH = Path("../data/processed/matches.csv")

raw = pd.read_csv(RAW_PATH, dtype="string", keep_default_na=False)
raw["source_row_number"] = range(len(raw))

print(f"Loaded {len(raw)} raw rows from {RAW_PATH}")

Loaded 49501 raw rows from ..\data\raw\international_results.csv


## Parse and sort match dates

Dates are parsed with an explicit `YYYY-MM-DD` format so malformed values
surface as `NaT` instead of being silently misparsed. Rows are then sorted by
`date` using a stable sort with `source_row_number` as the deterministic
tiebreaker, so matches played on the same day keep their original upstream
order every time this notebook runs.

In [2]:
matches = raw.copy()
matches["date"] = pd.to_datetime(
    matches["date"].str.strip(), format="%Y-%m-%d", errors="raise"
)
matches = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)

print(f"Date range: {matches['date'].min().date()} to {matches['date'].max().date()}")

Date range: 1872-11-30 to 2026-07-06


### Validation: date parsing and sort determinism

In [3]:
assert matches["date"].notna().all(), "every row must have a parsed date"
assert len(matches) == len(raw), "sorting must not drop or add rows"
assert matches["date"].is_monotonic_increasing, "rows must be sorted by date"

resorted = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)
assert matches["source_row_number"].tolist() == resorted["source_row_number"].tolist(), (
    "re-sorting the already-sorted table must be a no-op (deterministic order)"
)

same_day = matches[matches.duplicated(subset="date", keep=False)]
for _, group in same_day.groupby("date"):
    assert group["source_row_number"].is_monotonic_increasing, (
        "same-day matches must preserve original upstream order"
    )

print("Date parsing and deterministic sort validated.")

Date parsing and deterministic sort validated.


## Normalize team names and preserve originals

Team names are normalized (surrounding whitespace stripped, internal
whitespace collapsed) and passed through an explicit alias mapping before use
in any downstream feature or model. The unnormalized values are preserved in
`original_home_team` / `original_away_team` so every processed row remains
traceable to the exact upstream string, even if a future refresh or a
supplementary source (see `docs/data-sources.md`) introduces a name this
mapping does not yet cover.

The current snapshot's team names are already whitespace-clean and
single-form (verified below), so `TEAM_NAME_ALIASES` starts empty. The
mapping exists so a future snapshot revealing spelling variants (for example
an alternate name for the same team) can be corrected in one place without
rewriting the pipeline.

In [4]:
TEAM_NAME_ALIASES: dict[str, str] = {}


def normalize_team_name(name: str, aliases: dict[str, str] = TEAM_NAME_ALIASES) -> str:
    """Collapse whitespace and apply the explicit alias mapping."""
    normalized = " ".join(name.strip().split())
    return aliases.get(normalized, normalized)


matches["original_home_team"] = matches["home_team"]
matches["original_away_team"] = matches["away_team"]
matches["home_team"] = matches["home_team"].map(normalize_team_name)
matches["away_team"] = matches["away_team"].map(normalize_team_name)

print(f"Distinct normalized teams: {pd.unique(matches[['home_team', 'away_team']].values.ravel()).size}")

Distinct normalized teams: 336


### Validation: normalization and traceability

In [5]:
for column in ("home_team", "away_team"):
    assert (matches[column] == matches[column].str.strip()).all(), (
        f"{column} must not have leading/trailing whitespace"
    )
    assert (~matches[column].str.contains(r"\s{2,}", regex=True)).all(), (
        f"{column} must not contain repeated internal whitespace"
    )
    assert (matches[column] != "").all(), f"{column} must not be empty after normalization"

# Normalization must be idempotent.
assert matches["home_team"].map(normalize_team_name).equals(matches["home_team"])
assert matches["away_team"].map(normalize_team_name).equals(matches["away_team"])

# Original values must be preserved untouched for traceability.
assert matches["original_home_team"].equals(raw.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)["home_team"])
assert matches["original_away_team"].equals(raw.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)["away_team"])

print("Team name normalization and traceability validated.")

Team name normalization and traceability validated.


## Handle missing, duplicate, malformed, and impossible records

Documented rules applied in order:

1. **Missing scores.** Rows with `NA` for both scores are scheduled fixtures
   that have not been played yet (see `docs/data-sources.md`). They carry no
   outcome to learn from, so they are excluded from the processed table and
   the excluded count is reported. A row with only one score missing would be
   a malformed partial record; none exist in this snapshot, but the rule is
   the same: exclude it, since it cannot support a `result`.
2. **Duplicates.** Exact duplicate rows (identical on every raw column) are
   removed, keeping the first occurrence by `source_row_number` so the kept
   copy is deterministic.
3. **Malformed dates.** Already handled: date parsing above uses
   `errors="raise"`, so a malformed date fails the pipeline instead of being
   silently dropped or coerced.
4. **Malformed `neutral` flag.** Must be exactly `TRUE` or `FALSE`; anything
   else fails the pipeline rather than being guessed.
5. **Impossible scores.** Negative or non-integer scores are rejected.

In [6]:
before_dedupe = len(matches)
matches = matches.drop_duplicates(
    subset=[c for c in matches.columns if c not in ("source_row_number",)],
    keep="first",
)
duplicate_count = before_dedupe - len(matches)

missing_score_mask = (matches["home_score"] == "NA") | (matches["away_score"] == "NA")
partial_score_mask = (matches["home_score"] == "NA") ^ (matches["away_score"] == "NA")
assert not partial_score_mask.any(), "a row must have both scores or neither"

scheduled_fixture_count = int(missing_score_mask.sum())
matches = matches.loc[~missing_score_mask].copy()

assert matches["neutral"].isin({"TRUE", "FALSE"}).all(), "neutral must be TRUE or FALSE"
matches["neutral"] = matches["neutral"].map({"TRUE": True, "FALSE": False}).astype("boolean")

matches["home_score"] = pd.to_numeric(matches["home_score"], errors="raise").astype("Int64")
matches["away_score"] = pd.to_numeric(matches["away_score"], errors="raise").astype("Int64")
assert (matches["home_score"] >= 0).all() and (matches["away_score"] >= 0).all(), (
    "scores must be non-negative"
)

print(
    f"Removed {duplicate_count} exact duplicate rows and "
    f"{scheduled_fixture_count} scheduled fixtures without a final score. "
    f"{len(matches)} completed matches remain."
)

Removed 0 exact duplicate rows and 8 scheduled fixtures without a final score. 49493 completed matches remain.


### Validation: missing, duplicate, and malformed record handling

In [7]:
assert scheduled_fixture_count == 8, "expected 8 scheduled fixtures in this pinned snapshot"
assert (matches["home_score"].notna() & matches["away_score"].notna()).all(), (
    "every remaining row must have both final scores"
)
assert not matches.duplicated(
    subset=[c for c in matches.columns if c not in ("source_row_number",)]
).any(), "no exact duplicate rows may remain"
assert matches["neutral"].dtype == "boolean"
assert matches["neutral"].notna().all(), "neutral must never be null after conversion"
assert len(matches) == before_dedupe - duplicate_count - scheduled_fixture_count

print("Missing, duplicate, and malformed record handling validated.")

Missing, duplicate, and malformed record handling validated.


## Derive `result` and `goal_difference`

`result` is the match outcome from the home team's perspective (`H`/`D`/`A`),
and `goal_difference` is `home_score - away_score`. Both are derived purely
from already-known final scores, so they carry no information leakage risk.

In [8]:
matches["goal_difference"] = matches["home_score"] - matches["away_score"]
matches["result"] = pd.Series("D", index=matches.index, dtype="string")
matches.loc[matches["goal_difference"] > 0, "result"] = "H"
matches.loc[matches["goal_difference"] < 0, "result"] = "A"

print(matches["result"].value_counts())

result
H    24256
A    13981
D    11256
Name: count, dtype: Int64


### Validation: derived fields

In [9]:
assert set(matches["result"].unique()) <= {"H", "D", "A"}
assert (matches["goal_difference"] == matches["home_score"] - matches["away_score"]).all()
assert (matches.loc[matches["result"] == "H", "goal_difference"] > 0).all()
assert (matches.loc[matches["result"] == "A", "goal_difference"] < 0).all()
assert (matches.loc[matches["result"] == "D", "goal_difference"] == 0).all()
assert matches["result"].notna().all()
assert matches["goal_difference"].notna().all()

print("Derived result and goal_difference fields validated.")

Derived result and goal_difference fields validated.
